# 🏥 MedRAX Reproduction: Local Multimodal Vision (`qwen3-vl:4b-instruct`) on ChestAgentBench
### Reproducing "MedRAX: Medical Reasoning Agent for Chest X-ray" (ICML 2025)
This notebook performs **100% Local, Unlimited Multimodal Reproduction** of the MedRAX benchmark using **`qwen3-vl:4b-instruct`** via Ollama on Google Colab GPU.

| Component | Specification |
| :--- | :--- |
| **Benchmark** | **ChestAgentBench** (2,500 expert-curated clinical questions across 7 clinical axes from Eurorad) |
| **Vision & Agent Model** | **`qwen3-vl:4b-instruct`** via Ollama (Fast Non-Thinking Instruct VLM + Native Tool Calling) |
| **Cost & Rate Limits** | **100% Free, Zero API Keys, Zero Rate Limits, Never Blocked** (Runs 100% on Colab GPU) |
| **VRAM Optimization** | `~3.8 GB` (`qwen3-vl:4b-instruct` Q4_K_M) + `~8.5 GB` (4 MedRAX Tools in 16-bit) = **`~12.3 / 15.0 GB`** (Fits T4 15GB & L4 22.5GB) |
| **Agent Tools** | **CheXagent-2-3B VQA**, TorchXRayVision Classifier & Segmentation, Report Generator |
| **Storage** | Google Drive persistent cache (batch commits every 10 questions with `os.fsync`) |


## 1. ⚙️ Hardware & GPU Environment Diagnostics
Verify the Colab runtime environment (CUDA availability, GPU model, and VRAM).


In [ ]:
# Check GPU availability and specifications
import os
import sys
import torch

print(f"Python Version : {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Detected GPU   : {device_name} ({vram_gb:.2f} GB VRAM)")
    print("🚀 Local GPU ready for qwen3-vl:4b-instruct + MedRAX clinical vision tools!")
else:
    print("⚠️ WARNING: No GPU detected! Please enable GPU in Runtime -> Change runtime type.")

!nvidia-smi


## 2. 💾 Google Drive Mounting & Persistent Storage
Mount Google Drive to persist downloaded datasets, model checkpoints, and evaluation results.
Batch commits ensure that results are flushed directly to Google Drive every 10 questions.


In [ ]:
# Mount Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/MedRAX_Thesis')

PERSISTENT_DIRS = {
    "checkpoints": DRIVE_ROOT / "checkpoints",
    "data": DRIVE_ROOT / "data" / "chestagentbench",
    "results": DRIVE_ROOT / "results",
    "logs": DRIVE_ROOT / "logs",
}

for name, path in PERSISTENT_DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"📁 Verified persistent directory: {path}")

# Set environment variables
os.environ["MEDRAX_MODEL_DIR"] = str(PERSISTENT_DIRS["checkpoints"])
os.environ["BENCHMARK_DATA_DIR"] = str(PERSISTENT_DIRS["data"])
os.environ["MEDRAX_RESULTS_DIR"] = str(PERSISTENT_DIRS["results"])
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

print("\n✅ Environment variables configured successfully.")


## 3. 🐙 Clone Reproduction Repository from GitHub
Clone the reproduction suite from GitHub. If already present, pull the latest code updates.


In [ ]:
# Configure GitHub repository
import os
from pathlib import Path

REPO_URL = "https://github.com/Uyhuynh23/MedRAX-ChestAgentBench.git"
LOCAL_REPO_DIR = Path("/content/MedRAX-ChestAgentBench")

if not LOCAL_REPO_DIR.exists():
    print(f"Cloning {REPO_URL} into {LOCAL_REPO_DIR} ...")
    !git clone {REPO_URL} {LOCAL_REPO_DIR}
else:
    print(f"Repository exists. Pulling latest updates...")
    %cd {LOCAL_REPO_DIR}
    !git pull origin main

%cd {LOCAL_REPO_DIR}
!python scripts/setup_drive.py


## 4. 📦 Install Dependencies & Specialized Radiology Packages
Install PyTorch, LangChain, TorchXRayVision, and Transformers.


In [ ]:
%cd /content/MedRAX-ChestAgentBench

# Install project dependencies
!pip install -q -r requirements.txt
!pip install -q torchxrayvision

print("\n✅ All dependencies installed successfully.")


## 5. 👁️ Install Ollama & Pull Local Multimodal Model (`qwen3-vl:4b-instruct`)

Why **`qwen3-vl:4b-instruct`** via Ollama is ideal for MedRAX on Colab:
1. **Fast Instruct Variant (No Hidden Thinking Delay)**: Responds in ~2–4 seconds directly into `message.content` (unlike the default `qwen3-vl:4b` thinking tag which spends ~2 minutes generating hidden `<think>` tokens).
2. **Native Vision Encoder**: Directly reads chest X-ray images (`.jpg`, `.png`) unlike text-only `qwen2.5:3b`.
3. **Native Tool Calling (`{{ .Tools }}`)**: Supports LangChain `.bind_tools()` out of the box.
4. **VRAM Optimized**: Uses `OLLAMA_FLASH_ATTENTION=1` and `OLLAMA_KV_CACHE_TYPE=q8_0` to fit comfortably alongside MedRAX's 4 tools on a 15.0 GB T4 GPU.


In [ ]:
# -------------------------------------------------------------
# Install Ollama + Configure Low-VRAM Flash Attention + Pull qwen3-vl:4b-instruct
# -------------------------------------------------------------
import os
import time

# 1. Install zstd & pciutils so Ollama extracts and detects NVIDIA GPU properly
!apt-get update -qq && apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Configure memory-saving flags for Ollama (keeps VRAM usage ~3.8 GB on T4)
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q8_0"
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"

# 3. Start Ollama server in background with memory optimization flags
!pkill -f "ollama serve" || true
!OLLAMA_FLASH_ATTENTION=1 OLLAMA_KV_CACHE_TYPE=q8_0 OLLAMA_NUM_PARALLEL=1 OLLAMA_MAX_LOADED_MODELS=1 nohup ollama serve > /content/ollama.log 2>&1 &
time.sleep(5)

# 4. Pull qwen3-vl:4b-instruct (Fast non-thinking Instruct variant, ~3.3 GB)
!ollama pull qwen3-vl:4b-instruct
!ollama list

print("\n✅ qwen3-vl:4b-instruct is installed and serving locally at http://localhost:11434/v1")


In [ ]:
# -------------------------------------------------------------
# 🧪 Strict Visual Ground-Truth Proof & Tool-Calling Diagnostic
# -------------------------------------------------------------
# Draws a secret code ("7492") ONLY inside the image pixels (never in the prompt)
# and verifies that qwen3-vl:4b-instruct actually reads it with its Vision Encoder!
import time
import base64
import io
from PIL import Image, ImageDraw
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL_NAME = "qwen3-vl:4b-instruct"

# Draw secret visual code "7492" onto an image canvas
test_img = Image.new("RGB", (320, 120), color=(20, 20, 20))
draw = ImageDraw.Draw(test_img)
draw.rectangle([10, 10, 310, 110], outline=(220, 220, 220), width=3)
draw.text((40, 45), "SECRET CODE: 7492", fill=(255, 255, 255))

buf = io.BytesIO()
test_img.save(buf, format="JPEG", quality=95)
b64_img = base64.b64encode(buf.getvalue()).decode("utf-8")

print("=======================================================")
print(f" Running Strict Vision & Tool Proof ({MODEL_NAME})")
print("=======================================================\n")

# Test 1: Strict Visual Ground-Truth Proof (Model MUST read '7492' from pixels)
t0 = time.perf_counter()
resp_vis = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Read the exact 4-digit secret code written inside this image."},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64_img}"}}
        ]
    }],
    temperature=0.0,
    max_tokens=128,
)
t_vis = time.perf_counter() - t0
vis_txt = " ".join((resp_vis.choices[0].message.content or "").split())

if "7492" in vis_txt:
    print(f"  [Test 1: Multimodal Vision] ✅ PROVEN ({t_vis:.2f}s) -> Model read '7492' from image! Output: {vis_txt}")
else:
    print(f"  [Test 1: Multimodal Vision] ❌ FAILED ({t_vis:.2f}s) -> Expected '7492', got: '{vis_txt}'")

# Test 2: Native Tool Calling Support
t1 = time.perf_counter()
resp_tool = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Call chest_xray_classifier on /content/figure_1.jpg"}],
    tools=[{
        "type": "function",
        "function": {
            "name": "chest_xray_classifier",
            "description": "Classifies chest X-ray image for 18 pathologies",
            "parameters": {
                "type": "object",
                "properties": {"image_path": {"type": "string"}},
                "required": ["image_path"]
            }
        }
    }],
    temperature=0.0,
    max_tokens=256,
)
t_tool = time.perf_counter() - t1
t_calls = resp_tool.choices[0].message.tool_calls
if t_calls and t_calls[0].function.name == "chest_xray_classifier":
    print(f"  [Test 2: Native Tool Call ] ✅ PROVEN ({t_tool:.2f}s) -> Tool called: {t_calls[0].function.name}({t_calls[0].function.arguments})")
else:
    print(f"  [Test 2: Native Tool Call ] ❌ FAILED ({t_tool:.2f}s) -> No tool call detected.")


## 6. 📥 Download ChestAgentBench Dataset & Extract Figures
Download the **ChestAgentBench** benchmark dataset (`wanglab/chest-agent-bench`) from Hugging Face into Google Drive.
If already present on Google Drive, it will verify integrity and skip re-downloading.


In [ ]:
# Download / Verify ChestAgentBench in persistent Google Drive
%cd /content/MedRAX-ChestAgentBench
!python scripts/download_benchmark.py --data-dir "$BENCHMARK_DATA_DIR"

!ls -lh "$BENCHMARK_DATA_DIR"
!head -n 2 "$BENCHMARK_DATA_DIR/metadata.jsonl" | cut -c 1-120


## 7. 🔍 Check Saved Progress on Google Drive
Run the cell below anytime to check how many questions are already saved in Google Drive before starting or resuming a run.


In [ ]:
# -------------------------------------------------------------
# 📊 Check Saved Progress on Google Drive
# -------------------------------------------------------------
import json
from pathlib import Path

results_dir = Path(os.environ["MEDRAX_RESULTS_DIR"])
for run_name in ["qwen3vl_baseline_full", "qwen3vl_medrax_full"]:
    pred_file = results_dir / run_name / "predictions.jsonl"
    if pred_file.exists():
        lines = [json.loads(line) for line in open(pred_file, encoding="utf-8") if line.strip()]
        corr = sum(1 for r in lines if r.get("is_correct"))
        acc = (corr / len(lines) * 100) if lines else 0.0
        print(f"📁 {run_name:<24}: {len(lines):>4}/2500 questions saved | Running Acc: {acc:.2f}% ({corr}/{len(lines)})")
    else:
        print(f"📁 {run_name:<24}:    0/2500 questions saved (Not started yet)")


## 8. 🧪 Benchmark Execution & Evaluation (`qwen3-vl:4b-instruct` Local Vision)

Evaluate on **ChestAgentBench** with **`qwen3-vl:4b-instruct`** (100% Free, Local, Unlimited):
1. **Step 8.1a & 8.1b (Smoke Tests)**: Fast verification on 1–2 cases for both Baseline and MedRAX Agent modes.
2. **Option A (Full Baseline Multimodal)**: `qwen3-vl:4b-instruct` reads the chest radiographs directly with its native vision encoder and answers all 2,500 questions.
3. **Option B (Full MedRAX Agent Mode)**: `qwen3-vl:4b-instruct` reads the radiographs AND coordinates MedRAX's specialized clinical tools (**CheXagent-2-3B VQA**, TorchXRayVision classification & segmentation, report generator).


In [ ]:
# -------------------------------------------------------------
# 🧪 Step 8.1a: Smoke Test - Baseline Vision Mode on 2 Cases
# -------------------------------------------------------------
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py \
    --mode baseline \
    --model qwen3-vl:4b-instruct \
    --base-url "http://localhost:11434/v1" \
    --api-key "ollama" \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/smoke_test_qwen3vl_baseline" \
    --max-cases 2 \
    --reset


In [ ]:
# -------------------------------------------------------------
# 🧪 Step 8.1b: Smoke Test - MedRAX Agent Mode on 1 Case
# -------------------------------------------------------------
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py \
    --mode medrax \
    --model qwen3-vl:4b-instruct \
    --base-url "http://localhost:11434/v1" \
    --api-key "ollama" \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/smoke_test_qwen3vl_medrax" \
    --max-cases 1 \
    --reset


In [ ]:
# -------------------------------------------------------------
# 📊 Option A: Full Multimodal Baseline Evaluation (All 2,500 Questions)
# -------------------------------------------------------------
# qwen3-vl:4b-instruct reads chest radiographs directly with its native vision encoder.
# Batch commits every 10 questions with os.fsync directly to Google Drive.
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py \
    --mode baseline \
    --model qwen3-vl:4b-instruct \
    --base-url "http://localhost:11434/v1" \
    --api-key "ollama" \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/qwen3vl_baseline_full"
# Note: Add --reset ONLY if you want to clear previous progress and start from 0


In [ ]:
# -------------------------------------------------------------
# 🏥 Option B: Full MedRAX Agent Evaluation (All 2,500 Questions)
# -------------------------------------------------------------
# qwen3-vl:4b-instruct sees the radiographs + coordinates local clinical tools (CheXagent, TorchXRayVision).
# Batch commits every 10 questions with os.fsync directly to Google Drive.
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py \
    --mode medrax \
    --model qwen3-vl:4b-instruct \
    --base-url "http://localhost:11434/v1" \
    --api-key "ollama" \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/qwen3vl_medrax_full"
# Note: Add --reset ONLY if you want to clear previous progress and start from 0


## 9. 📊 Comparative Analytics: Baseline vs MedRAX Agent
Generate side-by-side comparison tables and accuracy breakdown charts across the 7 clinical axes:
* **Detection**, **Classification**, **Localization**, **Comparison**, **Relationship**, **Diagnosis**, **Characterization**


In [ ]:
# -------------------------------------------------------------
# ⚖️ Compare Baseline vs MedRAX Agent Performance
# -------------------------------------------------------------
%cd /content/MedRAX-ChestAgentBench

!python experiments/compare_runs.py \
    "$MEDRAX_RESULTS_DIR/qwen3vl_baseline_full/predictions.jsonl" \
    "$MEDRAX_RESULTS_DIR/qwen3vl_medrax_full/predictions.jsonl" \
    --output-dir "$MEDRAX_RESULTS_DIR/comparisons_qwen3vl"

# Display generated comparison plot
from IPython.display import Image, display
chart_path = Path(os.environ["MEDRAX_RESULTS_DIR"]) / "comparisons_qwen3vl" / "accuracy_breakdown.png"
if chart_path.exists():
    display(Image(str(chart_path)))


## 10. 🔄 Synchronize Results & Updates to GitHub
Push any notebook changes or evaluation summaries back to your GitHub repository.


In [ ]:
# Configure Git credentials in Colab
!git config --global user.name "Uy Huynh"
!git config --global user.email "huynhtanuy123@gmail.com"

# Check Git status
%cd /content/MedRAX-ChestAgentBench
!git status -s
